In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install -q transformers datasets accelerate seqeval evaluate
!pip install -q pillow scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import os
import random
import numpy as np

from PIL import Image

from sklearn.model_selection import train_test_split

import torch

from datasets import Dataset

from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/GOST_Project/layoutlm_dataset_2.json"

IMAGES_DIR = "/content/drive/MyDrive/GOST_Project/общие_фото_новинка"

MODEL_OUTPUT = "/content/drive/MyDrive/GOST_Project/layoutlm_model2"

In [ ]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Страниц:", len(data))

Страниц: 327


In [ ]:
labels = sorted(
    list(
        set(
            word["label"]
            for page in data
            for word in page["words"]
        )
    )
)

print(labels)
print(len(labels))

['BODY', 'FIGURE', 'FIGURE_CAPTION', 'FOOTNOTE', 'FORMULA', 'OTHER', 'REFERENCE', 'SECTION_HEADER', 'TABLE', 'TABLE_CAPTION', 'TITLE', 'TOC']
12


In [ ]:
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'BODY': 0, 'FIGURE': 1, 'FIGURE_CAPTION': 2, 'FOOTNOTE': 3, 'FORMULA': 4, 'OTHER': 5, 'REFERENCE': 6, 'SECTION_HEADER': 7, 'TABLE': 8, 'TABLE_CAPTION': 9, 'TITLE': 10, 'TOC': 11}


In [ ]:
train_data, test_data = train_test_split(
    data,
    test_size=0.15,
    random_state=42
)

train_data, val_data = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42
)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

Train: 235
Val: 42
Test: 50


In [ ]:
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

In [ ]:
from PIL import Image
import os

def normalize_bbox(bbox, width, height):

    return [
        int(1000 * bbox[0] / width),
        int(1000 * bbox[1] / height),
        int(1000 * bbox[2] / width),
        int(1000 * bbox[3] / height),
    ]


def prepare_examples(pages):

    dataset = []

    for page in pages:

        image_path = os.path.join(
            IMAGES_DIR,
            page["image"]
        )

        image = Image.open(image_path).convert("RGB")

        width, height = image.size

        words = []
        boxes = []
        labels = []

        for word in page["words"]:

            words.append(word["text"])

            boxes.append(
                normalize_bbox(
                    word["bbox"],
                    width,
                    height
                )
            )

            labels.append(
                label2id[word["label"]]
            )

        dataset.append(
            {
                "image": image,
                "words": words,
                "boxes": boxes,
                "labels": labels
            }
        )

    return dataset

In [ ]:
train_examples = prepare_examples(train_data)
val_examples = prepare_examples(val_data)
test_examples = prepare_examples(test_data)

print(len(train_examples))
print(len(val_examples))
print(len(test_examples))

235
42
50


In [ ]:
print(train_examples[0].keys())

print(train_examples[0]["words"][:5])
print(train_examples[0]["boxes"][:5])
print(train_examples[0]["labels"][:5])

dict_keys(['image', 'words', 'boxes', 'labels'])
['тензор; который возвращает нейронная сеть_', 'тензор; который содержит', 'данные', 'из датасета;', 'на котором происходит']
[[154, 122, 629, 144], [158, 153, 448, 175], [453, 157, 534, 171], [539, 156, 672, 172], [678, 156, 932, 172]]
[0, 0, 0, 0, 0]


In [ ]:
def encode_example(example):

    encoding = processor(
        images=example["image"],
        text=example["words"],
        boxes=example["boxes"],
        word_labels=example["labels"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    encoding = {
        k: v.squeeze()
        for k, v in encoding.items()
    }

    return encoding

In [ ]:
train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)
test_ds = Dataset.from_list(test_examples)

print(train_ds)

Dataset({
    features: ['image', 'words', 'boxes', 'labels'],
    num_rows: 235
})


In [ ]:
train_ds = train_ds.map(
    encode_example,
    remove_columns=train_ds.column_names
)

val_ds = val_ds.map(
    encode_example,
    remove_columns=val_ds.column_names
)

test_ds = test_ds.map(
    encode_example,
    remove_columns=test_ds.column_names
)

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
train_ds

Dataset({
    features: ['labels', 'input_ids', 'attention_mask', 'bbox', 'pixel_values'],
    num_rows: 235
})

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.dense.weight    | MISSING | 
classifier.out_proj.weight | MISSING | 
classifier.out_proj.bias   | MISSING | 
classifier.dense.bias      | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):

        cur_preds = []
        cur_labels = []

        for p_i, l_i in zip(pred, lab):

            if l_i == -100:
                continue

            cur_preds.append(
                id2label[p_i]
            )

            cur_labels.append(
                id2label[l_i]
            )

        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    num_train_epochs=15,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=10,

    save_total_limit=2,

    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_ds,
    eval_dataset=val_ds,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.106569,0.626960,0.927973,0.824405,0.873128,0.881129
2,0.095029,0.697248,0.953043,0.815476,0.878909,0.889302
3,0.048654,0.593800,0.930533,0.857143,0.892332,0.894502
4,0.111022,0.648694,0.949919,0.875000,0.910922,0.914562
5,0.004467,0.694737,0.962393,0.837798,0.895784,0.902675
6,0.008712,0.750051,0.951096,0.839286,0.891700,0.898217
7,0.091157,0.698791,0.984375,0.843750,0.908654,0.907875
8,0.003626,0.684467,0.979021,0.833333,0.900322,0.898960
9,0.056494,0.719866,0.984238,0.836310,0.904264,0.904903
10,0.004046,0.743274,0.975524,0.830357,0.897106,0.898217


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: REFERENCE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: OTHER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE_CAPTION seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BODY seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: REFERENCE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: OTHER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE_CAPTION seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BODY seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: REFERENCE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: OTHER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE_CAPTION seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BODY seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: REFERENCE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: OTHER seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: TABLE_CAPTION seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BODY seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1770, training_loss=0.11445837368778253, metrics={'train_runtime': 2110.9334, 'train_samples_per_second': 1.67, 'train_steps_per_second': 0.838, 'total_flos': 935645874278400.0, 'train_loss': 0.11445837368778253, 'epoch': 15.0})

In [ ]:
trainer.save_model(MODEL_OUTPUT)

processor.save_pretrained(MODEL_OUTPUT)

In [ ]:
metrics = trainer.evaluate(test_ds)

print(metrics)

In [ ]:
import os

os.listdir(MODEL_OUTPUT)

['checkpoint-913', 'checkpoint-1245']

In [ ]:
!ls -lh /content/drive/MyDrive/GOST_Project/layoutlm_model/checkpoint-1245

total 1.5G
-rw------- 1 root root 1.5K Jun 14 23:46 config.json
-rw------- 1 root root 481M Jun 14 23:46 model.safetensors
-rw------- 1 root root 961M Jun 14 23:46 optimizer.pt
-rw------- 1 root root  15K Jun 14 23:46 rng_state.pth
-rw------- 1 root root 1.4K Jun 14 23:46 scaler.pt
-rw------- 1 root root 1.5K Jun 14 23:46 scheduler.pt
-rw------- 1 root root  29K Jun 14 23:46 trainer_state.json
-rw------- 1 root root 5.1K Jun 14 23:46 training_args.bin


In [ ]:
!du -sh /content/drive/MyDrive/GOST_Project/layoutlm_model

3.3G	/content/drive/MyDrive/GOST_Project/layoutlm_model


In [ ]:
import os

os.listdir(MODEL_OUTPUT)

['checkpoint-913',
 'checkpoint-1245',
 'config.json',
 'model.safetensors',
 'training_args.bin',
 'tokenizer_config.json',
 'tokenizer.json',
 'processor_config.json']

In [ ]:
import shutil

shutil.make_archive(
    "/content/layoutlm_model_backup",
    "zip",
    MODEL_OUTPUT
)

'/content/layoutlm_model_backup.zip'